# 02 - Đọc kết quả thực nghiệm đã xác minh

Notebook này **không tự huấn luyện hoặc mở test**. Chỉ liệt kê manifest tại `results/runs`, chọn run `evaluated` mới nhất theo `created_at` có model/checksum hợp lệ, rồi hiển thị JSON CV/test và hình đã xác minh checksum. Không chọn run theo điểm test. Thiếu model/kết quả thì báo rõ, không dự đoán giả.

Nguồn dữ liệu: Water Potability, Aditya Kadiwal, Kaggle: https://www.kaggle.com/datasets/adityakadiwal/water-potability. Nguồn/checksum thực tế được ghi trong manifest dữ liệu và run. Kết quả chỉ phản ánh nhãn Potability của bộ dữ liệu, không phải chứng nhận nước uống. Demo học thuật, không thay thế kiểm nghiệm chất lượng nước.

CV dùng tập phát triển để chọn cấu hình; test chỉ để báo cáo sau khi chốt. Không sửa seed/tham số dựa trên các kết quả dưới đây. Ví dụ 2D ở cuối là dữ liệu tổng hợp độc lập, không phải kết quả thực nghiệm thật.

In [ ]:
from pathlib import Path
import json
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'src' / 'config.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Không tìm thấy project root chứa src/config.py từ cwd/parents.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display, Markdown, Image, JSON
from src.pipeline import load_pipeline, sha256_file
from src.train import validate_run_id
from src.gaussian_nb_numpy import GaussianNaiveBayes
print('Chế độ chỉ đọc kết quả; không gọi train/evaluate.')

In [ ]:
run_rows, eligible_runs = [], []
for manifest_path in sorted((PROJECT_ROOT / 'results' / 'runs').glob('*.json')):
    try:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        run_id = validate_run_id(manifest['run_id'])
        if manifest_path.stem != run_id:
            raise ValueError('Tên manifest không khớp run_id.')
        status = 'Chưa đánh giá test; không chọn'
        if manifest.get('stage') == 'evaluated':
            model_path = PROJECT_ROOT / 'artifacts' / run_id / 'pipeline.json'
            pipeline = load_pipeline(model_path, expected_sha256=manifest['model_sha256'])
            if pipeline.run_id != run_id:
                raise ValueError('Model không khớp run_id.')
            eligible_runs.append(manifest)
            status = 'Model và checksum hợp lệ'
        run_rows.append({'run_id': run_id, 'stage': manifest.get('stage'),
                         'created_at': manifest.get('created_at'), 'status': status})
    except (ValueError, OSError, KeyError, TypeError) as error:
        run_rows.append({'run_id': manifest_path.stem, 'status': 'Từ chối: ' + str(error)})
if run_rows:
    display(pd.DataFrame(run_rows))
else:
    print('Chưa có manifest run; không có kết quả thật để hiển thị.')
selected_run = max(eligible_runs, key=lambda item: (item.get('created_at', ''), item['run_id'])) if eligible_runs else None
if selected_run is None:
    display(Markdown('**Chưa có run evaluated với model/checksum hợp lệ.** Không tự train, không mở test và không tạo dự đoán thay thế.'))
else:
    print('Run evaluated mới nhất có model/checksum hợp lệ:', selected_run['run_id'])

## Kết quả thật từ run đã chọn
Mỗi file phải nằm trong thư mục metrics/figures của đúng run và có SHA-256 khớp `files` trong manifest trước khi đọc/hiển thị. JSON CV/test giữ nguyên số liệu; hình không được tính lại. Checksum sai hoặc file thiếu sẽ được báo và không hiển thị. Hình EDA nếu có vẫn thuộc dev; confusion matrix và ROC là đánh giá test đã lưu.

In [ ]:
if selected_run is not None:
    run_id = selected_run['run_id']
    file_checksums = selected_run.get('files', {})
    metrics_root = (PROJECT_ROOT / 'results' / 'metrics' / run_id).resolve()
    figures_root = (PROJECT_ROOT / 'results' / 'figures' / run_id).resolve()
    required_json = [f'results/metrics/{run_id}/{name}' for name in
                     ('cv_results.json', 'test_metrics.json', 'baseline_test_metrics.json')]
    figure_files = sorted(name for name in file_checksums if name.endswith('.png'))
    for relative_path in required_json + figure_files:
        try:
            path = (PROJECT_ROOT / relative_path).resolve()
            allowed_root = figures_root if path.suffix == '.png' else metrics_root
            if not path.is_relative_to(allowed_root):
                raise ValueError('Đường dẫn nằm ngoài thư mục kết quả của run.')
            if relative_path not in file_checksums or sha256_file(path) != file_checksums[relative_path]:
                raise ValueError('Checksum file thiếu hoặc không khớp.')
            display(Markdown('### Đã xác minh: ' + path.name))
            if path.suffix == '.png':
                display(Image(filename=str(path)))
            else:
                display(JSON(json.loads(path.read_text(encoding='utf-8'))))
        except (ValueError, OSError, KeyError, TypeError) as error:
            print('Không hiển thị', relative_path, ':', error)
else:
    print('Bỏ qua phần kết quả thật: chưa có run đủ điều kiện.')

## Kiểm chứng lý thuyết 2D độc lập (không phải kết quả Kaggle)
Dữ liệu tổng hợp: lớp 0 có (0,0), (2,2); lớp 1 có (4,4), (6,6). Đáp án tính tay: mỗi lớp có 2 mẫu, prior = 1/2; trung bình lớp 0 là (1,1), lớp 1 là (5,5); phương sai MLE mỗi chiều mỗi lớp là 1 vì chia tổng bình phương độ lệch 2 cho n=2. Trung bình toàn bộ là (3,3), phương sai toàn cục mỗi chiều là (9+1+1+9)/4 = 5.

Chọn var_smoothing = 10^-9, variance_floor = 10^-12 cho ví dụ: epsilon = 5×10^-9, phương sai hiệu dụng v = 1.000000005. Tại (3,3), hai lớp có cùng log-score ln(1/2) - ln(2πv) - 4/v, nên posterior chính xác (0.5,0.5), nhãn hòa là 0. Tại (1,1), chênh log-score lớp 0 so với lớp 1 là 16/v; posterior lớp 1 xấp xỉ 0.00000011253517, nhãn 0. Đây là đáp án độc lập chuẩn bị trước, không lấy output mô hình làm expected.

Cell sau chỉ gọi bản tự cài trong src, không sao chép công thức huấn luyện/dự đoán. Sai số posterior cho phép 10^-12. Không liên hệ các điểm 2D này với thông số nước thực tế.

In [ ]:
example_model = GaussianNaiveBayes(var_smoothing=1e-9, variance_floor=1e-12)
example_model.fit([[0, 0], [2, 2], [4, 4], [6, 6]], [0, 0, 1, 1])
example_state = example_model.to_state()
assert example_state['class_counts'] == [2, 2]
assert example_state['class_priors'] == [0.5, 0.5]
assert example_state['means'] == [[1.0, 1.0], [5.0, 5.0]]
assert example_state['variances_mle'] == [[1.0, 1.0], [1.0, 1.0]]
assert example_state['global_variances_mle'] == [5.0, 5.0]
assert abs(example_state['epsilon'] - 5e-9) < 1e-20
example_probabilities = example_model.predict_proba([[3, 3], [1, 1]])
assert abs(example_probabilities[0][0] - 0.5) < 1e-12
assert abs(example_probabilities[0][1] - 0.5) < 1e-12
assert abs(example_probabilities[1][1] - 0.00000011253517) < 1e-12
assert example_model.predict([[3, 3], [1, 1]]) == [0, 0]
print('Ví dụ tổng hợp 2D đã khớp đáp án tính tay; KHÔNG phải kết quả thực nghiệm thật.')
display(pd.DataFrame(example_probabilities, columns=['P(lớp 0)', 'P(lớp 1)'],
                     index=['Ví dụ (3,3): hòa', 'Ví dụ (1,1)']))